In [ ]:
!pip install shap transformers librosa datasets pingouin

In [ ]:
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import shap
import pingouin as pg
from scipy.stats import pearsonr, spearmanr
from datasets import load_dataset, Audio, DatasetDict
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
TARGET_IDX = 35
n_segments = 50
N_SAMPLES = 2000
SEEDS = [0, 1, 2, 3, 4]
TOP_K = 10

#Φόρτωση Repo και Dataset

In [ ]:
repo_id = "VasilisAsim/data2vec-finetuned-TESS_full"
model = AutoModelForAudioClassification.from_pretrained(repo_id)
feature_extractor = AutoFeatureExtractor.from_pretrained(repo_id)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

In [ ]:
def load_tess_speaker_independent():
    dataset = load_dataset("AbstractTTS/TESS", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))

    def get_speaker(example):
        filename = example["audio"]["path"]
        if "OAF" in filename:
            return "OAF"
        elif "YAF" in filename:
            return "YAF"
        return "unknown"

    train_dataset = dataset.filter(lambda x: get_speaker(x) == "OAF")
    test_dataset = dataset.filter(lambda x: get_speaker(x) == "YAF")
    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    return DatasetDict({"train": train_dataset, "test": test_dataset})

dataset = load_tess_speaker_independent()
test_dataset = dataset["test"]

In [ ]:
sample = test_dataset[TARGET_IDX]
audio = sample["audio"]["array"]
sampling_rate = sample["audio"]["sampling_rate"]
true_label = sample["emotion"]

inputs = feature_extractor(audio, sampling_rate=sampling_rate, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}
with torch.no_grad():
    probabilities = torch.softmax(model(**inputs).logits, dim=-1)
predicted_class = probabilities.argmax(-1).item()

print("True label:", true_label)
print("Predicted:", model.config.id2label[predicted_class])

###Segmentation και predict

In [ ]:
segments = np.array_split(audio, n_segments)

def reconstruct_audio(mask):
    """Ανασυνθέτει το σήμα κρατώντας τα segments με mask=1 και μηδενίζοντας τα υπόλοιπα."""
    return np.concatenate(
        [seg if m == 1 else np.zeros_like(seg) for seg, m in zip(segments, mask)]
    )


def predict(mask_batch):
    """Πιθανότητες κλάσεων για κάθε μάσκα"""
    outputs = []
    for mask in mask_batch:
        audio_input = reconstruct_audio(mask)
        inputs = feature_extractor(audio_input, sampling_rate=sampling_rate, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            logits = model(**inputs).logits
        outputs.append(torch.softmax(logits, dim=-1).squeeze().cpu().numpy())
    return np.array(outputs)


background = np.zeros((1, n_segments))   # baseline: όλα τα segments σβησμένα

### KernelSHAP σε πολλαπλά seeds

In [ ]:
def normalise_attribution(attr):
    """Απόλυτες τιμές κανονικοποιημένες ώστε το άθροισμα να είναι 1"""
    a = np.abs(attr)
    return a / (a.sum() + 1e-10)


shap_runs = []
for seed in SEEDS:
    np.random.seed(seed)
    explainer = shap.KernelExplainer(predict, background)
    shap_values = explainer.shap_values(np.ones((1, n_segments)), nsamples=N_SAMPLES)
    shap_runs.append(normalise_attribution(shap_values[0, :, predicted_class]))
    print(f"Seed {seed}: ολοκληρώθηκε")

shap_runs = np.array(shap_runs)            # (n_runs, N_SEGMENTS)
n_runs = len(SEEDS)
assert not np.allclose(shap_runs[0], shap_runs[1]), "Τα runs είναι ίδια: το seed δεν επηρεάζει το sampling"

### Ομοιότητα ανά ζεύγος επαναλήψεων (Pearson, Spearman, Jaccard top-k)

In [ ]:
def jaccard_topk(a, b, k=TOP_K):
    """Jaccard των k κορυφαίων segments δύο χαρτών."""
    top_a, top_b = set(np.argsort(a)[-k:]), set(np.argsort(b)[-k:])
    return len(top_a & top_b) / len(top_a | top_b)


rows = []
for i, j in itertools.combinations(range(n_runs), 2):
    rows.append({
        "run_1": SEEDS[i], "run_2": SEEDS[j],
        "pearson": pearsonr(shap_runs[i], shap_runs[j])[0],
        "spearman": spearmanr(shap_runs[i], shap_runs[j])[0],
        "jaccard": jaccard_topk(shap_runs[i], shap_runs[j]),
    })

df_pairs = pd.DataFrame(rows)
print(df_pairs.round(3).to_string(index=False))
print("\nΜέσος όρος ± τυπική απόκλιση πάνω στα ζεύγη:")
print(df_pairs[["pearson", "spearman", "jaccard"]].agg(["mean", "std"]).round(3))

### Διασπορά και συντελεστής μεταβλητότητας ανά segment

In [ ]:
mean_seg = shap_runs.mean(axis=0)
std_seg = shap_runs.std(axis=0, ddof=1)
var_seg = shap_runs.var(axis=0, ddof=1)

top_idx = np.argsort(mean_seg)[-TOP_K:]             # τα TOP_K πιο σημαντικά segments
weighted_cv = std_seg.sum() / mean_seg.sum()        # Σ std / Σ mean (mean.sum() = 1)
median_cv_top = np.median(std_seg[top_idx] / mean_seg[top_idx])

print(f"Μέση διασπορά ανά segment: {var_seg.mean():.3e}")
print(f"Σταθμισμένο CV: {weighted_cv:.3f}")
print(f"Διάμεσο CV στα {TOP_K} κορυφαία segments: {median_cv_top:.3f}")

plt.figure(figsize=(12, 4))
plt.bar(np.arange(n_segments), mean_seg, yerr=std_seg, capsize=2)
plt.xlabel("Segment")
plt.ylabel("Μέση τιμή SHAP ± τυπική απόκλιση")
plt.title(f"Σταθερότητα KernelSHAP σε {n_runs} seeds (idx={TARGET_IDX})")
plt.tight_layout()
plt.show()

### Intraclass Correlation Coefficient (ICC1, ICC1k)

In [ ]:
icc_df = pd.DataFrame(
    [{"segment": s, "run": SEEDS[r], "value": shap_runs[r, s]}
     for r in range(n_runs) for s in range(n_segments)]
)
icc = pg.intraclass_corr(data=icc_df, targets="segment", raters="run", ratings="value")
print(icc.to_string(index=False))                 # ολόκληρος ο πίνακας, για έλεγχο
print("\nΕπιλεγμένα (εναλλάξιμα runs):")
print(icc.iloc[[0, 3]].to_string(index=False))    # ICC1 και ICC1k

In [ ]:
### ICC μόνο στα ενεργά segments (μη μηδενικά σε τουλάχιστον ένα run)

active = np.where(shap_runs.max(axis=0) > 0)[0]
print(f"Ενεργά segments: {len(active)} από {n_segments}")

icc_act_df = pd.DataFrame(
    [{"segment": s, "run": SEEDS[r], "value": shap_runs[r, s]}
     for r in range(n_runs) for s in active]
)
icc_act = pg.intraclass_corr(data=icc_act_df, targets="segment", raters="run", ratings="value")
print(icc_act.iloc[[0, 3]].to_string(index=False))